# Step 4: de novo assembly attempt

In step 3 the genome was built **with** a reference. **De novo** assembly builds it **without** one. The program looks for reads that overlap each other and joins them into longer and longer pieces, called **contigs**, until, with luck, one contig is the whole genome.

This is useful when no good reference exists. For SARS-CoV-2 a reference does exist, so step 3's consensus is the one the use case carries forward. This notebook tries de novo assembly anyway, to show what it gives with this kind of data.

**Expect it to fail or to give fragments.** The reason is below.

What it does, in order:

1. Explains the choice of tool.
2. Runs the assembler on the clean reads of one run.
3. Reports what came out.
4. If anything came out, places the pieces on the reference to see which part of the genome each covers.
5. Compares with step 3.

Run the cells one at a time, from top to bottom, and read the output of each before moving on.

## Which assembler, and why not SPAdes

**SPAdes** is the usual choice for viral assembly, but it is built for short Illumina reads. Its own documentation says it should not be used when the only reads are Oxford Nanopore. These files are Nanopore, so SPAdes does not apply.

**Flye** is the standard assembler for Nanopore reads, so that is what is used here.

**Why this data is hard to assemble.** The reads come from **amplicon** sequencing. The virus genome (30,000 bases) was copied in about 30 overlapping pieces, each about 1,000 to 1,200 bases. Every read is one whole piece. Flye needs reads that overlap by about 1,000 bases to join them. Neighbouring pieces here overlap by about 100. So Flye has very little to join with.

This is a known limit of de novo assembly on tiled-amplicon data, not a fault of this notebook.

## 1. Imports

- `subprocess` runs another program from Python.
- `os`, `sys` and `shutil` are used once, to check the tools can be found.
- `Path` builds file and folder paths.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

## 2. Your input and the folders

- `ACCESSION`: the same as in the earlier steps.
- `REFERENCE_ACCESSION`: the reference used in step 3. It is used here only to see where the assembled pieces fall.
- `MAX_FILES`: how many files to assemble. Flye takes about two minutes per file, so the default is 1. Raise it to assemble more.
- `GENOME_SIZE`: roughly how long the genome is. `30k` means 30,000 bases. Flye needs an estimate.
- `THREADS`: how many processor threads the tools may use.

In [ ]:
ACCESSION = "PRJEB40277"
REFERENCE_ACCESSION = "MN908947.3"
MAX_FILES = 1
GENOME_SIZE = "30k"
THREADS = 8

DATA_FOLDER = Path("data") / ACCESSION
CLEAN_FOLDER = DATA_FOLDER / "clean"
ASSEMBLY_FOLDER = DATA_FOLDER / "assembly"
REFERENCE_FILE = Path("data") / "reference" / f"{REFERENCE_ACCESSION}.fasta"

ASSEMBLY_FOLDER.mkdir(parents=True, exist_ok=True)

clean_files = sorted(CLEAN_FOLDER.glob("*.clean.fastq.gz"))[:MAX_FILES]

for clean_file in clean_files:
    print(clean_file.name)

## 3. Check the tools can be found

The first line makes sure this notebook can find programs installed in the `ls-use-case` environment.

**Look at the output.** Both lines should show a location that contains `ls-use-case`. If you see `None`, choose the `ls-use-case` kernel (top right) and run the cell again.

In [ ]:
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

for tool in ["flye", "minimap2"]:
    print(tool, "->", shutil.which(tool))

## 4. Run Flye

`--nano-raw` tells Flye the reads are raw Nanopore reads. `--out-dir` is where it writes its results, one folder per run in `assembly/`.

The cell keeps going whether Flye succeeds or not. `result.returncode` is `0` if Flye finished without an error, and another number if it stopped. The cell prints the last lines of Flye's own log so you can read why.

Flye prints a lot while it works, so the cell hides it and shows only the end.

**This takes about two minutes per file.**

**Look at the output.** Flye is expected to stop with `No disjointigs were assembled`. A disjointig is Flye's first rough piece. None were made because the reads overlap too little.

Lines worth explaining:

- `result.stdout + result.stderr` joins the two output streams of the program. Flye writes its log to one of them.
- `.splitlines()[-4:]` is the last four lines.

In [ ]:
for clean_file in clean_files:
    run = clean_file.name.split(".")[0]
    out_folder = ASSEMBLY_FOLDER / run

    command = [
        "flye",
        "--nano-raw", str(clean_file),
        "--out-dir", str(out_folder),
        "--genome-size", GENOME_SIZE,
        "--threads", str(THREADS),
    ]

    result = subprocess.run(command, capture_output=True, text=True)

    print(run, "Flye exit code:", result.returncode)
    for line in (result.stdout + result.stderr).splitlines()[-4:]:
        print("    ", line)

## 5. What came out?

If Flye succeeded it wrote `assembly.fasta` in the run's folder. This cell looks for it and lists each contig and its length.

A FASTA file has a name line starting with `>`, then the sequence on the lines below. The cell reads it line by line and adds up the length of each contig.

**Look at the output.** One contig of about 29,900 bases would be a complete genome. Several short ones are fragments.

In [ ]:
assemblies = {}

for clean_file in clean_files:
    run = clean_file.name.split(".")[0]
    assembly_file = ASSEMBLY_FOLDER / run / "assembly.fasta"

    if not assembly_file.exists():
        print(run, "no assembly was made")
        continue

    lengths = {}
    with open(assembly_file) as input_file:
        for line in input_file:
            if line.startswith(">"):
                name = line[1:].strip()
                lengths[name] = 0
            else:
                lengths[name] = lengths[name] + len(line.strip())

    assemblies[run] = assembly_file

    print(run, "contigs:", len(lengths))
    for name, length in lengths.items():
        print("    ", name, length, "bases")

## 6. Where do the pieces fall on the genome?

This section only does something if section 5 found contigs.

It maps each contig to the reference with `minimap2`, using `-x asm5` (the setting for sequences that are very similar to the reference). The output is in PAF format, one line per match. Four columns are printed:

- `columns[0]`: the contig
- `columns[1]`: the contig's length
- `columns[7]` and `columns[8]`: where the match starts and ends on the reference

In [ ]:
if len(assemblies) == 0:
    print("No contigs to place. Nothing to do here.")

for run, assembly_file in assemblies.items():
    paf_file = ASSEMBLY_FOLDER / run / "contigs_on_reference.paf"

    subprocess.run(
        ["minimap2", "-x", "asm5", "-o", str(paf_file), str(REFERENCE_FILE), str(assembly_file)],
        check=True,
        stderr=subprocess.DEVNULL,
    )

    print(run)
    with open(paf_file) as input_file:
        for line in input_file:
            columns = line.split("\t")
            print("    ", columns[0], "length", columns[1], "covers reference positions", columns[7], "to", columns[8])

## 7. Which result goes forward?

Compare with step 3.

| | Step 3: reference-based | Step 4: de novo |
|---|---|---|
| Input needed | A reference genome | Only the reads |
| Result on this data | One sequence of about 29,900 bases | See above |
| Used for the next step | **Yes** | No |

Step 3's consensus goes forward. De novo assembly is the right tool when the virus is new or very different from anything known, or when the reads are long enough to overlap. Neither is true here.

**What would make de novo work.** Sequencing without amplicons (for example shotgun or metagenomic reads), or whole-genome reads that are tens of thousands of bases long.

Nothing more to run. Step 5 is next.